# exp00: Penerimaan Dataset

Memeriksa ketiga good_acceptable, identitas lintas sesi, proporsi kelas, serta kesiapan input sebelum release disahkan. Notebook ini tidak melatih classifier.


## Akses Proyek

Menentukan root lokal atau Google Drive. Semua notebook memakai paket batch-02 yang sama; hanya lokasi proyek yang perlu disesuaikan.


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

PROJECT_ROOT = None
if PROJECT_ROOT is None:
    if IN_COLAB:
        PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
    else:
        PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                             if (p / 'models' / 'batch-02').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Isi PROJECT_ROOT dengan folder proyek yang memuat models/batch-02.')
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
BATCH_DIR = PROJECT_ROOT / 'models' / 'batch-02'
if not (BATCH_DIR / 'shared' / 'batch02').is_dir():
    raise RuntimeError('Salin folder batch-02 lengkap, termasuk shared dan requirements.txt.')
sys.path.insert(0, str(BATCH_DIR / 'shared'))


## Environment

Menyiapkan dependency inti sebelum memuat modul data. Instalasi hanya dilakukan bila versi berbeda. Dependency neural dan TabPFN bersifat opsional dan tidak mengunduh checkpoint. Jika instalasi mengubah modul yang sudah dimuat, restart runtime sekali lalu jalankan dari awal.


In [ ]:
from batch02.runtime import ensure_environment

INSTALL_DEPENDENCIES = IN_COLAB
INSTALL_OPTIONAL = False
OPTIONAL_GROUPS = ()
environment_status = ensure_environment(
    BATCH_DIR, install=INSTALL_DEPENDENCIES,
    optional=OPTIONAL_GROUPS if INSTALL_OPTIONAL else (),
)
print(environment_status)


## Konfigurasi

Menetapkan sumber data dan output tanpa mengunci jumlah peserta atau distribusi kelas. Seed mengikuti protokol batch-02. Tidak ada kewajiban menjalankan mode smoke sebelum evaluasi penelitian.


In [ ]:
from batch02.contracts import Config

EXPERIMENT = 'exp00'
DATASET_ROOT = None
RESUME_DIR = None
config = Config(PROJECT_ROOT, dataset_root=DATASET_ROOT)
print('Dataset:', config.dataset)
print('Output:', config.batch / 'outputs' / EXPERIMENT)


## Pemeriksaan Dataset

Membaca ketiga CSV dan allowlist, memeriksa label/interval/fitur, lalu menampilkan proporsi aktual serta gate yang belum disetujui. Dataset sekarang boleh diperiksa; pemeriksaan ini tidak menyatakan data tambahan sudah masuk.


In [ ]:
from batch02.runner import readiness

readiness_report = readiness(config)
display(readiness_report['dataset'])
display(readiness_report['release'])
display(readiness_report['capabilities'])


## Registry Identitas Lintas Sesi

Tinjau person_id yang berasal dari metadata. Jika seseorang mempunyai beberapa sesi, gunakan person_id yang sama melalui PERSON_ALIASES. Konfirmasi registry hanya setelah identitasnya diperiksa; kolom ini tidak menjadi fitur model.


In [ ]:
from batch02.runner import registry_draft

session_registry = registry_draft(config)
PERSON_ALIASES = {}
CONFIRM_IDENTITY_REGISTRY = False
for row in session_registry['sessions']:
    row['person_id'] = PERSON_ALIASES.get(row['session_id'], row['person_id'])
    row['confirmed'] = CONFIRM_IDENTITY_REGISTRY
display(session_registry)


## Pengesahan Release

Setelah data tambahan digabung dan proporsi disetujui, isi ID sesi tambahan serta nama penanggung jawab. Pengesahan memvalidasi registry/split dan membekukan snapshot. Flag di bawah sengaja belum disetujui; menjalankan notebook sekarang tidak boleh mengaktifkan training.


In [ ]:
from batch02.contracts import freeze_release

APPROVE_RELEASE = False
PROPORTIONS_APPROVED = False
ADDITIONAL_SESSION_IDS = []
APPROVED_BY = ''

if APPROVE_RELEASE:
    dataset_release = freeze_release(
        config, session_registry, ADDITIONAL_SESSION_IDS, APPROVED_BY,
        proportions_approved=PROPORTIONS_APPROVED,
    )
    display(dataset_release)
else:
    print('Release belum disahkan; training tetap diblokir.')


## Cache Aliran Sensor

Sesudah release disahkan, jalankan persiapan cache ini di lokal yang memiliki sumber mentah. Cache memuat seluruh grid, termasuk endpoint tanpa target tunggal, dan dicocokkan terhadap CSV. Untuk Colab, salin snapshot beserta folder streams; tidak perlu mengunggah rekaman mentah.


In [ ]:
from batch02.stream_adapter import prepare_stream_cache

PREPARE_STREAM_CACHE = False
RAW_SESSION_ROOT = PROJECT_ROOT / 'sessions' / 'saved_session'
if PREPARE_STREAM_CACHE:
    display(prepare_stream_cache(config, RAW_SESSION_ROOT))
else:
    print('Persiapan cache tidak dijalankan.')
